In [ ]:
from lib.drivers import *
from serial.tools import list_ports
import time
from matplotlib import pyplot as plt

In [ ]:
%matplotlib widget
plt.close("all")

In [ ]:
ports = list_ports.comports()
for port in ports:
    print(port.device)

In [ ]:
port = "COM14"

elettrometro = ElettrometroKeithley(port)

def query(cmd):
    response = elettrometro.query(cmd)
    if response is not None and response != "":
        print(response)

query(elettrometro.commands["identify"])

#elettrometro.close()

In [ ]:
elettrometro.reset()
time.sleep(0.1)
elettrometro.set_zero_check(False)
time.sleep(0.1)


In [ ]:
elettrometro.init_current_reading()

In [ ]:
currents=[]
times=[]
voltages=[]


vs = np.arange(-1, 1, step=0.05)

print(f"Leggendo {len(vs)} volte, con tensioni tra {np.min(vs):.2f} e {np.max(vs):.2f}")

#elettrometro.set_autorange(False)
#elettrometro.set_manual_range(1e-9)

for v in vs:
    elettrometro.set_source_voltage(v)
    t=elettrometro.get_fresh_reading()
    corrente, tempo=elettrometro.parse_resistance_reading(t)
    currents.append(corrente)
    times.append(tempo)
    voltages.append(v)
    print(f"V={v:.2f}", end = ",")
    time.sleep(0.1)

elettrometro.set_source_voltage(0)
elettrometro.get_fresh_reading()
currents=np.array(currents)
voltages=np.array(voltages)


In [ ]:
plt.figure()
plt.subplot(2, 1, 1)

plt.plot(currents)
plt.title("Correnti")

plt.subplot(2, 1, 2)

plt.plot(voltages)
plt.title("Tensioni")

plt.tight_layout()
plt.show()

In [ ]:
res=voltages/currents

[print(f"{r}, ", end="") for r in res]
print()

mo = np.log10(np.mean(res))
print(f"Resistenza media: {np.mean(res)/10**np.floor(mo):.2f} * 10^{int(np.floor(mo))}")


In [ ]:
with open("elettrometro_test.txt", "w") as file:
    file.write("Tensione (V), Corrente (A)\n")
    for v, c in zip(voltages, currents):
        file.write(f"{v}, {c}\n")

In [ ]:
plt.figure()

a,b, =np.polyfit(currents,voltages,1)

plt.plot(currents,voltages)
plt.plot(currents, voltages, "x")

plt.plot(currents, a*currents+b, label=f"fit: V={a:.2e}*I + {b:.2e}")

plt.axvline(0, color="k")
plt.axhline(0, color="k")
plt.grid()
print(a,b)
plt.show()

In [ ]:
elettrometro.close()